# 属性传递、HLO 编辑与 cost model

对应 R09/R10/R11。运行环境为 CPU，保留 **VERSION-SKEW**。本 Notebook 执行 native HLO 解析、cost analysis 和属性操作，并复查捕获的数值；没有 TPU 后端编译或运行。
详见 [attributes-and-cost.md](attributes-and-cost.md) 与 [roofline.md](roofline.md)。

In [1]:
from pathlib import Path
import sys, json
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "upstream-sources.lock").is_file())
sys.path.insert(0, str(root / "research/jax-stack"))
from verify_attributes import verify
capture = root / "artifacts/jax-stack/attributes-cost-002"
result = verify(capture)
print("artifacts verified:", result["artifact_count"])
for case in result["cases"]:
    print(case["case"], "tagged HLO instructions:", case["tagged_hlo_instructions"], "FLOPs:", case["cost"]["flops"])

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


artifacts verified: 60
plain tagged HLO instructions: 0 FLOPs: 384.0
context tagged HLO instructions: 1 FLOPs: 384.0
value tagged HLO instructions: 1 FLOPs: 384.0
call tagged HLO instructions: 1 FLOPs: 384.0
grad-call-same tagged HLO instructions: 2 FLOPs: 792.0
grad-call-drop tagged HLO instructions: 1 FLOPs: 792.0


## 公开 metadata API 与默认成本可同时使用

这里重新执行同形状 matmul。`research_flops=999999` 是一个自定义标签，默认模型仍按形状计算 384 FLOPs。
一个描述可并行维度的提示也不会自动改变完整计算的工作量；若希望算每块成本，需要消费提示并定义分块语义。

In [2]:
import jax, jax.numpy as jnp, numpy as np
from jax.experimental.xla_metadata import set_xla_metadata
with np.load(capture / "inputs.npz", allow_pickle=False) as inputs:
    a, w = inputs["a"], inputs["w"]
def marked(a, w):
    with set_xla_metadata(research_flops=999999):
        return jnp.matmul(a, w, precision=jax.lax.Precision.HIGHEST)
lowered = jax.jit(marked).lower(a, w)
compiled = lowered.compile()
actual = np.asarray(compiled(a, w))
np.testing.assert_allclose(actual, a.astype(np.float64) @ w.astype(np.float64), rtol=2e-5, atol=2e-5)
assert 'research_flops="999999"' in lowered.compiler_ir("hlo").as_hlo_text()
assert compiled.cost_analysis()["flops"] == 384
print(compiled.cost_analysis())

{'bytes accessed1{}': 192.0, 'utilization1{}': 1.0, 'bytes accessedout{}': 96.0, 'utilization0{}': 1.0, 'bytes accessed': 416.0, 'flops': 384.0, 'bytes accessed0{}': 128.0}


## Native HLO 属性 getter/setter

下面只修改内存中的独立 HloModule，不写回源码或既有 capture。它演示私有 native API；任意图改写仍须验证 shape、effects、alias 等约束。

In [3]:
from jaxlib import _hlo
from jax._src.lib import _jax
from jax._src import xla_bridge
module = _hlo.hlo_module_from_text((capture / "plain/exported-hlo.txt").read_text())
dot = next(i for c in module.computations() for i in c.instructions() if i.opcode == _hlo.HloOpcode.kDot)
assert dot.get_frontend_attribute("notebook_tag") is None
dot.set_frontend_attribute("notebook_tag", "in-memory-only")
assert dot.get_frontend_attribute("notebook_tag") == "in-memory-only"
print(dot.to_string())
print(_jax.hlo_module_cost_analysis(xla_bridge.get_backend("cpu"), module))

%dot_general.1 = f32[4,6]{1,0} dot(%a.1, %w.1), lhs_contracting_dims={1}, rhs_contracting_dims={0}, operand_precision={highest,highest}, frontend_attributes={notebook_tag="in-memory-only"}
{'flops': 384.0, 'bytes accessed': 416.0, 'utilization0{}': 1.0, 'utilization1{}': 1.0, 'bytes accessed0{}': 128.0, 'bytes accessed1{}': 192.0, 'bytes accessedout{}': 96.0}


## 已执行的 HLO 改写与未知成本

捕获中的 add→subtract 已分别编译执行；第一单元重新解析 opcode，并复算 NumPy 参考。
CPU reference analyzer 对外层 TPU custom-call 返回 -1：这是未知成本，不能得到目标设备的 roofline。

In [4]:
before = np.load(capture / "hlo-edit/before-output.npy", allow_pickle=False)
after = np.load(capture / "hlo-edit/after-output.npy", allow_pickle=False)
with np.load(capture / "inputs.npz", allow_pickle=False) as inputs:
    np.testing.assert_allclose(before - after, 2 * inputs["b"], rtol=2e-5, atol=2e-5)
print(result["hlo_edit"]["results"])
print(result["opaque_custom_call"]["scope"])
print(result["opaque_custom_call"]["cost"])
assert result["opaque_custom_call"]["roofline_available"] is False

[{'case': 'before', 'max_absolute_error': 3.683228411155426e-08}, {'case': 'after', 'max_absolute_error': 7.337873109136694e-08}]
CPU reference analyzer applied to TPU-targeted IR; no libtpu cost analysis or device execution.
{'flops': -1.0, 'bytes accessed': -1.0, 'optimal_seconds': -1.0, 'utilization0{}': 1.0, 'utilization1{}': 1.0, 'bytes accessed0{}': -1.0, 'bytes accessed1{}': -1.0, 'bytes accessedout{}': -1.0}


## 已有模型键：latency_metadata

这是一项在部分 GPU latency estimator 中有消费逻辑的键。先在新 CPU 进程复查标签传递和成本 API，再阅读固定源码中的消费者；CPU 成本不等于 GPU NodeCost。详见 [模型调用链](latency-model.md)。

In [5]:
import subprocess
from datetime import datetime, timezone
latency_capture = root / "artifacts/jax-stack" / ("latency-notebook-" + datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S%f"))
command = [str(root / ".venv/bin/python"), "-B", str(root / "research/jax-stack/latency_metadata_probe.py"), "--output", str(latency_capture)]
run = subprocess.run(command, cwd=root, text=True, capture_output=True, timeout=180)
if run.returncode:
    raise RuntimeError(run.stdout + run.stderr)
print(run.stdout.strip())
from verify_latency_metadata import verify as verify_latency
latency_result = verify_latency(latency_capture)
for case in latency_result["cases"]:
    print(case["case"], repr(case["normalized_label"]), case["cpu_compiled_cost"]["flops"], case["cpu_compiled_cost"]["bytes accessed"])
print("verified artifacts:", latency_result["artifact_count"])

{"capture": "latency-notebook-20260914163236908786", "cases": 8, "qualifiers": ["VERSION-SKEW"]}


plain None 384.0 416.0
integer '30000' 384.0 416.0
string '30000' 384.0 416.0
zero '0' 384.0 416.0
negative '-1' 384.0 416.0
fractional '30000.5' 384.0 416.0
invalid 'slow' 384.0 416.0
int64-overflow '9223372036854775808' 384.0 416.0
verified artifacts: 45


In [6]:
contract = json.loads((root / "research/jax-stack/latency-model-contract.json").read_text())
print(contract["evidence_level"], contract["source_revision"])
print("Parser input unit:", contract["parser"]["input_unit"])
for reader in contract["direct_readers"]:
    print(reader["source_entry"], reader["opcode_scope"], reader["lookup_priority"])
assert contract["runtime_evidence"]["gpu_estimator_executed"] is False
assert contract["upstream_parser_test"]["executed"] is False
print("GPU estimator、C++ parser 测试与目标 TPU 均未由此 Notebook 执行")

SOURCE-ONLY 496bd4bd49db9ecbffd85da630b49c860b724604
Parser input unit: nanoseconds
xla.gpu-node-cost custom-call after nop check nop first; then metadata for custom-call; otherwise approximate fallback
xla.sol-node-cost all opcodes metadata before opcode/table/fusion branches
GPU estimator、C++ parser 测试与目标 TPU 均未由此 Notebook 执行


`30000` 的字符串能在 CPU HLO 中保留，不代表实测耗时 30 µs。非法或溢出标签被 CPU 接受，也不是 GPU parser 验证通过。可并行维度驱动的真实分块仍需要业务语义、正确的 cost consumer 和目标运行验证。